# 검증된 원자료 기준 지역·연도 추세 시각화

- 이슈: #45
- 구조환경지표: 2016~2025년 검증본 21개 지표
- 계획예산: 2016~2024년 시행계획의 당해 명목 계획예산
- 분석 단위: 17개 시도. 전국 행은 QA·추세 비교 기준으로만 사용한다.
- 결측은 보간하거나 0으로 대체하지 않으며 그래프에서 선이 끊기도록 유지한다.
- 급등락 표시는 오류 확정이 아니라 원자료 재확인 후보(IQR 기준)다.


## 1. 공통 설정

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / ".git").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("현재 실행 위치의 상위 경로에서 Git 저장소를 찾지 못했습니다.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.features.trend_eda import (  # noqa: E402
    build_structural_region_summary,
    prepare_budget_trends,
    render_structural_region_report,
    reshape_structural_indicators,
)
from src.visualization.plots import save_figure  # noqa: E402
from src.visualization.trends import (  # noqa: E402
    plot_budget_overview,
    plot_budget_region_small_multiples,
    plot_region_small_multiples,
    plot_structural_indicator_overview,
)

MAPPING_PATH = repo_root / "data" / "lookup" / "시도_지역코드_매핑.csv"
STRUCTURAL_PATH = (
    repo_root / "data" / "interim" / "구조환경지표_검증" / "구조환경지표_21개_검증본.csv"
)
BUDGET_PATH = (
    repo_root
    / "data"
    / "processed"
    / "analysis"
    / "2016-2024_시도별_계획예산_합계출산율_기초패널.csv"
)
OUTPUT_DIR = repo_root / "data" / "processed" / "eda" / "지역연도_추세"
FIGURE_DIR = repo_root / "notebooks" / "results" / "20260727_지역연도_추세"

region_order = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")["지역"].tolist()
assert len(region_order) == 17
print("저장소 루트:", repo_root)
print("시도:", region_order)

## 2. 구조환경지표 long 패널 및 QA

In [ ]:
structural_wide = pd.read_csv(STRUCTURAL_PATH, encoding="utf-8-sig")
structural_long = reshape_structural_indicators(
    structural_wide,
    expected_regions=region_order,
)

structural_regions = structural_long.loc[~structural_long["지역"].eq("전국")]
structural_qa = pd.DataFrame(
    {
        "항목": [
            "long 패널 행 수",
            "세부지표 수",
            "시도 수",
            "지역×연도×지표 중복",
            "17개 시도 원자료 결측 셀",
            "급등락 후보 셀",
        ],
        "값": [
            len(structural_long),
            structural_long["세부지표"].nunique(),
            structural_regions["지역"].nunique(),
            int(structural_long.duplicated(["지역", "연도", "세부지표"]).sum()),
            int((~structural_regions["실측여부"]).sum()),
            int(structural_regions["급등락후보"].fillna(False).sum()),
        ],
    }
)
display(structural_qa)

In [ ]:
structural_missing_summary = structural_regions.groupby(
    ["대영역", "세부영역", "세부지표"], as_index=False
).agg(
    전체셀=("실측여부", "size"),
    실측셀=("실측여부", "sum"),
)
structural_missing_summary["결측셀"] = (
    structural_missing_summary["전체셀"] - structural_missing_summary["실측셀"]
)
structural_missing_summary["결측률_pct"] = (
    structural_missing_summary["결측셀"].div(structural_missing_summary["전체셀"]).mul(100)
)

structural_outliers = structural_regions.loc[
    structural_regions["급등락후보"].fillna(False),
    ["지역", "연도", "대영역", "세부영역", "세부지표", "측정값", "전년대비변화", "검증상태"],
].sort_values(["세부지표", "연도", "지역"])

display(structural_missing_summary.sort_values("결측률_pct", ascending=False))
display(structural_outliers.head(30))

## 3. 구조환경지표 그래프 생성

전국 공표·산출값이 있으면 이를 기준선으로 사용하고, 근로시간처럼 전국값이 없으면 17개 시도 중앙값과 IQR을 표시한다.

In [ ]:
figure_records = []
structural_figure_dir = FIGURE_DIR / "구조환경지표"
structural_overview_dir = structural_figure_dir / "지표별_요약"
structural_regional_dir = structural_figure_dir / "17개시도_추세"
indicator_order = structural_wide["세부지표"].drop_duplicates().tolist()

for index, indicator in enumerate(indicator_order, start=1):
    stem = f"{index:02d}_{indicator}"

    overview = plot_structural_indicator_overview(
        structural_long,
        indicator=indicator,
    )
    overview_path = structural_overview_dir / f"{stem}_요약"
    save_figure(overview, overview_path)
    plt.close(overview)
    figure_records.append(
        {"구분": "구조환경지표 요약", "세부지표": indicator, "경로": str(overview_path)}
    )

    regional = plot_region_small_multiples(
        structural_long,
        indicator=indicator,
        region_order=region_order,
    )
    regional_path = structural_regional_dir / f"{stem}_17개시도"
    save_figure(regional, regional_path)
    plt.close(regional)
    figure_records.append(
        {"구분": "구조환경지표 지역", "세부지표": indicator, "경로": str(regional_path)}
    )

print(f"구조환경지표 그래프 세트: {len(figure_records)}개")

## 4. 계획예산 추세 및 QA

#53에서 생성한 검증 기초패널을 재사용한다. 금액은 실질화·인구 보정을 하지 않은 명목 계획예산이며 실제 집행액이 아니다.

In [ ]:
if not BUDGET_PATH.exists():
    raise FileNotFoundError(
        f"#53 기초패널이 없습니다. 먼저 20260726 기초패널 생성 노트북을 실행하세요: {BUDGET_PATH}"
    )

budget_base = pd.read_csv(BUDGET_PATH, encoding="utf-8-sig")
budget_trends = prepare_budget_trends(
    budget_base,
    expected_regions=region_order,
)
budget_outliers = budget_trends.loc[
    budget_trends["급등락후보"],
    [
        "지역",
        "연도",
        "당해계획예산_백만원",
        "전년대비증감률_pct",
        "원자료_누락주의",
    ],
].sort_values(["연도", "지역"])

budget_annual_summary = budget_trends.groupby("연도", as_index=False).agg(
    **{"17개시도_합계_백만원": ("당해계획예산_백만원", "sum")},
    시도_중앙값_백만원=("당해계획예산_백만원", "median"),
    급등락후보수=("급등락후보", "sum"),
    원자료누락주의수=("원자료_누락주의", lambda values: int(values.notna().sum())),
)
display(budget_annual_summary)
display(budget_outliers)

In [ ]:
budget_figure_dir = FIGURE_DIR / "계획예산"

budget_overview = plot_budget_overview(budget_trends)
budget_overview_path = budget_figure_dir / "계획예산_전국합계_지역분포_증감률_기본계획기간"
save_figure(budget_overview, budget_overview_path)
plt.close(budget_overview)
figure_records.append(
    {"구분": "계획예산 요약", "세부지표": pd.NA, "경로": str(budget_overview_path)}
)

budget_regional = plot_budget_region_small_multiples(
    budget_trends,
    region_order=region_order,
)
budget_regional_path = budget_figure_dir / "계획예산_17개시도_추세"
save_figure(budget_regional, budget_regional_path)
plt.close(budget_regional)
figure_records.append(
    {"구분": "계획예산 지역", "세부지표": pd.NA, "경로": str(budget_regional_path)}
)

print(f"전체 그래프 세트: {len(figure_records)}개")

## 5. 분석용 표와 그래프 목록 저장

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
figure_manifest = pd.DataFrame(figure_records)
figure_manifest["PNG"] = figure_manifest["경로"].astype(str) + ".png"
figure_manifest["PDF"] = figure_manifest["경로"].astype(str) + ".pdf"
figure_manifest = figure_manifest.drop(columns="경로")

structural_region_summary = build_structural_region_summary(
    structural_long,
    region_order=region_order,
)
regional_report_path = repo_root / "reports" / "20260727_구조환경지표별_17개시도_상세결과.md"
regional_report_path.write_text(
    render_structural_region_report(structural_region_summary),
    encoding="utf-8",
)

outputs = {
    "구조환경지표_long.csv": structural_long,
    "구조환경지표_결측요약.csv": structural_missing_summary,
    "구조환경지표_급등락후보.csv": structural_outliers,
    "구조환경지표_지역별상세.csv": structural_region_summary,
    "계획예산_지역연도_추세.csv": budget_trends,
    "계획예산_연도요약.csv": budget_annual_summary,
    "계획예산_급등락후보.csv": budget_outliers,
    "그래프_목록.csv": figure_manifest,
}
for file_name, frame in outputs.items():
    path = OUTPUT_DIR / file_name
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    print(path.relative_to(repo_root), frame.shape)
print(regional_report_path.relative_to(repo_root), structural_region_summary.shape)

display(figure_manifest)

## 해석상 주의사항

- 전국값과 17개 시도 단순평균은 혼용하지 않는다. 근로시간은 전국 공표값이 없어 시도 중앙값을 참고선으로 썼다.
- 명목 계획예산 합계는 전국 정부 총예산이 아니라 17개 시도 시행계획 세부사업의 관측 가능한 당해예산 합계다.
- 강원·전남 등 원자료 상세 누락 주의 지역은 실제 규모보다 과소대표될 수 있다.
- IQR 급등락 후보는 단위별 분포가 다른 지표의 재확인 목록이며 오류 확정값이 아니다.
- 제3차·제4차 기본계획 배경 구분은 기술적 시기 비교용이며 정책의 인과효과를 의미하지 않는다.
- 이 단계에서는 보간, 실질화, 대상인구 보정, 가중치 적용, 회귀분석을 수행하지 않는다.
